# 05 — Promote to Strong Validation

## What this notebook does
Collects shortlisted candidates from notebooks 03 and 04, applies minimum
score thresholds, produces a clean **promotion bundle** (machine-readable shortlist
with provenance), and writes a plain-language handoff protocol for the next
validation lane.

## What decision it helps make
> "Here is my documented shortlist and rationale. These candidates are ready
> for stronger validation. This is where I stop spending cheap compute."

## What it cannot prove
- That any candidate binds the target
- That the shortlist will survive a stronger lane
- Anything about potency, selectivity, or activity

---

> **Why this notebook should NOT be the first step:**
>
> Strong validation lanes (AlphaFold2, Boltz-1, PyRosetta, external docking)
> are slow and/or expensive. Running them without first calibrating the lane
> (notebook 02) and filtering through cheap triage (notebooks 03–04) wastes
> compute and produces results you cannot interpret.
>
> The intermediate cookbook phrase applies here:
> *"Use cheap compute for exploration. Use expensive compute for validation."
> Do not invert that order.*

## Free vs Paid Colab

This notebook is **free Colab compatible**. It only reads previous outputs
and writes text/JSON files.

The notebook also includes a section that generates inputs for common strong-validation
lanes (AlphaFold2 Colab, Boltz-1 local, PyRosetta). Running those lanes themselves
is out of scope here — they are separate notebooks or environments.

| Step | Free | Paid |
|------|------|------|
| Load results and build shortlist | Yes | Yes |
| Write promotion bundle | Yes | Yes |
| Generate AF2/Boltz input files | Yes | Yes |
| Run AF2 or Boltz validation | No (too slow) | Possible |
| Run PyRosetta target-decoy | No (not installed) | Possible |

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "biopython"])
print("Dependencies ready.")

In [ ]:
import sys, pathlib

# ── Environment detection ─────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    for _p in [
        pathlib.Path('/content/peptide-cookbooks-internal/colab-basics'),
        pathlib.Path('/content/colab-basics'),
    ]:
        if _p.exists():
            COOKBOOK_DIR = _p
            break
    else:
        raise RuntimeError(
            "Cookbook not found. Clone the repo first:\n"
            "  !git clone <your-repo-url> /content/peptide-cookbooks-internal"
        )
    WORKSPACE_DIR = pathlib.Path('/content/workspace')
else:
    COOKBOOK_DIR = pathlib.Path('..').resolve()
    WORKSPACE_DIR = COOKBOOK_DIR / 'workspace'

# ============================================================
# CONFIGURATION
# ============================================================

# Compute tier — controls size caps only. No GPU lane in this cookbook.
COMPUTE_TIER = "free"  # "free" or "paid"

MIN_SCORE_THRESHOLD = 0.50  # minimum composite score to promote
MAX_PROMOTED = 8             # cap; more than ~10 means you haven't been selective

FORCE_INCLUDE = [
    # {"sequence": "YOURSEQ", "label": "expert_pick_01", "reason": "literature comparison"},
]

TARGET_SPEC_PATH   = WORKSPACE_DIR / "target_spec.json"
PANEL_INTERP_PATH  = WORKSPACE_DIR / "reference_panel" / "panel_interpretation.txt"
SINGLE_EVAL_PATH   = WORKSPACE_DIR / "single_eval" / "eval_result.json"
SAR_META_PATH      = WORKSPACE_DIR / "sar_scan" / "sar_scan_meta.json"
SAR_SHORTLIST_PATH = WORKSPACE_DIR / "sar_scan" / "sar_shortlist.csv"
OUTPUT_DIR         = WORKSPACE_DIR / "promotion"

print(f"Environment: {'Colab' if IN_COLAB else 'local'}")
print(f"Min score: {MIN_SCORE_THRESHOLD}, max promoted: {MAX_PROMOTED}")
print(f"Panel gate file: {PANEL_INTERP_PATH}")

In [ ]:
# Setup
import sys, json
import csv

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(COOKBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(COOKBOOK_DIR))

from shared.target_utils import load_target_spec

if not TARGET_SPEC_PATH.exists():
    raise FileNotFoundError(
        f"Target spec not found at {TARGET_SPEC_PATH}. Run notebook 01 first."
    )

target_spec = load_target_spec(TARGET_SPEC_PATH)
print(f"Target: {target_spec['pdb_id']} chain {target_spec['chain_id']}")

# ── Panel calibration gate ──────────────────────────────────────────────────
# This is enforced, not advisory. A promotion bundle produced without a passing
# calibration check is misleading: the shortlist scores are meaningless if the
# lane cannot distinguish signal from noise.
if not PANEL_INTERP_PATH.exists():
    raise RuntimeError(
        f"Panel calibration file not found at {PANEL_INTERP_PATH}. "
        "Run notebook 02 (reference_panel_check) and confirm it passes "
        "before promoting any candidates."
    )

panel_interp_text = PANEL_INTERP_PATH.read_text()
status_line = next(
    (l.strip() for l in panel_interp_text.splitlines()
     if l.strip().lower().startswith("panel status:")),
    None,
)
if status_line is None:
    raise RuntimeError(
        "Could not parse panel status from panel_interpretation.txt. "
        "Re-run notebook 02 to regenerate that file."
    )

panel_status = status_line.split(":", 1)[1].strip().lower()
if panel_status == "fail":
    raise RuntimeError(
        f"Reference panel check returned FAIL (see {PANEL_INTERP_PATH}). "
        "Promotion is blocked. Fix your target spec (notebook 01 pocket hints) "
        "and re-run notebook 02 until the panel status is 'pass' or 'degraded'."
    )

print(f"Panel calibration status: {panel_status} — gate passed.")

## Step 1 — Collect Previous Results

Load outputs from notebooks 03 and 04.

In [ ]:
candidates = []
sources = []

# From notebook 03 (single peptide eval)
if SINGLE_EVAL_PATH.exists():
    single_result = json.loads(SINGLE_EVAL_PATH.read_text())
    candidates.append({
        "label": single_result.get("candidate_label", "eval_candidate"),
        "sequence": single_result["candidate_sequence"],
        "composite_score": float(single_result.get("composite_score", 0)),
        "source_notebook": "03_single_peptide_eval",
        "delta_vs_seed": None,
    })
    sources.append("notebook_03")
    print(f"Loaded 1 candidate from notebook 03")
else:
    print(f"notebook 03 result not found at {SINGLE_EVAL_PATH} — skipping")

# From notebook 04 (SAR scan shortlist)
if SAR_SHORTLIST_PATH.exists():
    with open(SAR_SHORTLIST_PATH) as f:
        sar_rows = list(csv.DictReader(f))
    for row in sar_rows:
        candidates.append({
            "label": row.get("label", "sar_variant"),
            "sequence": row.get("sequence", ""),
            "composite_score": float(row.get("composite_score", 0)),
            "source_notebook": "04_small_sar_scan",
            "delta_vs_seed": float(row.get("delta_vs_seed", 0)),
        })
    sources.append("notebook_04")
    print(f"Loaded {len(sar_rows)} candidates from notebook 04 SAR shortlist")
else:
    print(f"SAR shortlist not found at {SAR_SHORTLIST_PATH} — skipping")

# Add force-include candidates
for fi in FORCE_INCLUDE:
    candidates.append({
        "label": fi.get("label", "force_include"),
        "sequence": fi["sequence"],
        "composite_score": None,
        "source_notebook": "manual",
        "delta_vs_seed": None,
        "force_include_reason": fi.get("reason", ""),
    })
if FORCE_INCLUDE:
    print(f"Added {len(FORCE_INCLUDE)} force-include candidates")

print(f"\nTotal candidates collected: {len(candidates)}")

## Step 2 — Score Force-Include Candidates and Apply Threshold

In [ ]:
from shared.scoring_utils import score_peptide

# Score any candidates without a score (force-includes)
for c in candidates:
    if c['composite_score'] is None and c['sequence']:
        r = score_peptide(c['sequence'], target_spec)
        c['composite_score'] = r['composite_score']
        c['heuristic_interpretation'] = r['interpretation']

# Deduplicate by sequence
seen_seqs = set()
unique_candidates = []
for c in candidates:
    if c['sequence'] not in seen_seqs:
        seen_seqs.add(c['sequence'])
        unique_candidates.append(c)
print(f"After deduplication: {len(unique_candidates)} unique candidates")

# Apply threshold and rank
above_threshold = [
    c for c in unique_candidates
    if (c.get('composite_score') or 0) >= MIN_SCORE_THRESHOLD
    or c.get('source_notebook') == 'manual'
]
below_threshold = [
    c for c in unique_candidates
    if (c.get('composite_score') or 0) < MIN_SCORE_THRESHOLD
    and c.get('source_notebook') != 'manual'
]

above_threshold.sort(key=lambda x: x.get('composite_score') or 0, reverse=True)
shortlist = above_threshold[:MAX_PROMOTED]

print(f"Above threshold ({MIN_SCORE_THRESHOLD}): {len(above_threshold)}")
print(f"Below threshold: {len(below_threshold)}")
print(f"Promotion shortlist (capped at {MAX_PROMOTED}): {len(shortlist)}")

print("\nShortlist:")
print(f"{'Label':<25} {'Score':>7} {'Source':<25} {'Sequence'}")
print("-" * 85)
for c in shortlist:
    sc = f"{c.get('composite_score', 0):.4f}" if c.get('composite_score') is not None else "N/A"
    print(f"{c['label']:<25} {sc:>7} {c['source_notebook']:<25} {c['sequence']}")

if below_threshold:
    print(f"\nExcluded (below threshold):")
    for c in below_threshold[:5]:
        sc = f"{c.get('composite_score', 0):.4f}"
        print(f"  {c['label']:<25} {sc:>7} {c['sequence']}")
    if len(below_threshold) > 5:
        print(f"  ... and {len(below_threshold)-5} more")

## Step 3 — Write Promotion Bundle

The promotion bundle is a machine-readable JSON file containing the shortlist,
provenance, and caveats. Use this to hand off work to another notebook, environment,
or collaborator.

In [ ]:
import datetime

bundle = {
    "created": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "cookbook": "colab-basics",
    "target": {
        "pdb_id": target_spec["pdb_id"],
        "chain_id": target_spec["chain_id"],
        "pocket_description": target_spec.get("pocket_description", ""),
        "mechanism_hypothesis": target_spec.get("mechanism_hypothesis", ""),
    },
    "shortlist": shortlist,
    "excluded": [
        {"label": c['label'], "sequence": c['sequence'],
         "composite_score": c.get('composite_score'),
         "reason": f"Below threshold {MIN_SCORE_THRESHOLD}"}
        for c in below_threshold
    ],
    "provenance": {
        "sources": sources + (["manual"] if FORCE_INCLUDE else []),
        "lane": "heuristic_composition",
        "min_score_threshold": MIN_SCORE_THRESHOLD,
        "max_promoted": MAX_PROMOTED,
    },
    "claims_level": 1,
    "claims_ladder_note": (
        "Level 1 (cheap heuristic): claim is 'this sequence class is worth escalating'. "
        "Not a binding claim. Not a potency claim. Not a selectivity claim."
    ),
    "caveats": [
        "Heuristic lane only. Score does not imply binding affinity.",
        "No target-decoy validation has been performed.",
        "Reference panel calibration must have passed (notebook 02) for this shortlist to be meaningful.",
        "Candidates should be validated with a structure-prediction or physics-based lane before synthesis.",
    ],
    "next_step_options": [
        "AlphaFold2 multimer — AF2 Colab notebook (structure-prediction based, models full complex)",
        "Boltz-1 — local or cloud GPU, good for protein-peptide complexes",
        "PyRosetta FlexPepDock or target-decoy — see intermediate-linux-gpu cookbook",
        "HADDOCK web server — peptide docking with restraints (free academic access)",
    ],
    "next_step_note": (
        "Small-molecule docking tools (AutoDock Vina, Smina) are not appropriate here. "
        "They do not model backbone flexibility and their scoring functions were not "
        "parameterised for peptides of 8-20 residues. Use structure-prediction-based "
        "or peptide-specific docking methods instead."
    ),
}

bundle_path = OUTPUT_DIR / "shortlist.json"
bundle_path.write_text(json.dumps(bundle, indent=2))
print(f"Promotion bundle saved to {bundle_path}")

## Step 4 — Write Human-Readable Promotion Summary

In [ ]:
lines = [
    "# Promotion Summary",
    "",
    f"**Generated:** {bundle['created']}",
    f"**Cookbook:** {bundle['cookbook']}",
    "",
    "## Target",
    f"- PDB: {bundle['target']['pdb_id']}, Chain: {bundle['target']['chain_id']}",
    f"- Pocket: {bundle['target']['pocket_description']}",
    f"- Hypothesis: {bundle['target']['mechanism_hypothesis']}",
    "",
    "## Shortlisted Candidates",
    "",
    "| # | Label | Sequence | Score | Source |",
    "|---|-------|----------|-------|--------|",
]
for i, c in enumerate(shortlist, 1):
    sc = f"{c.get('composite_score', 0):.4f}" if c.get('composite_score') is not None else "N/A"
    lines.append(f"| {i} | {c['label']} | `{c['sequence']}` | {sc} | {c['source_notebook']} |")

lines += [
    "",
    "## Excluded Candidates",
    f"{len(below_threshold)} candidates were below the threshold of {MIN_SCORE_THRESHOLD}.",
    "",
    "## Claims Level",
    "",
    bundle['claims_ladder_note'],
    "",
    "## Caveats",
    "",
]
for c in bundle['caveats']:
    lines.append(f"- {c}")

lines += [
    "",
    "## Next Steps",
    "",
    "Choose a strong-validation lane from the options below and run these candidates through it.",
    "Do NOT treat the heuristic scores above as a final ranking.",
    "",
]
for opt in bundle['next_step_options']:
    lines.append(f"- {opt}")

lines += [
    "",
    "## Handoff Protocol",
    "",
    "1. Confirm reference panel check passed (notebook 02). If it did not, do not promote.",
    "2. Provide the sequences from `shortlist.json` to your chosen validation lane.",
    "3. Run the reference positive control through the same validation lane.",
    "4. A candidate is worth considering only if it performs comparably to or better than",
    "   the reference in the validation lane — not just in this heuristic lane.",
    "5. Record the validation results in your project results folder with provenance.",
]

summary_text = "\n".join(lines)
summary_path = OUTPUT_DIR / "promotion_summary.md"
summary_path.write_text(summary_text)
print(summary_text)
print(f"\nSaved to {summary_path}")

## Step 5 — Generate FASTA File for Strong-Validation Input

In [ ]:
# Write a FASTA file with shortlisted peptides — useful as input for
# AF2 multimer, Boltz-1, or other structure-prediction tools.
#
# For AF2 multimer / Boltz-1: provide the receptor chain + each peptide as
# separate sequences in the same FASTA. The receptor chain is loaded from
# the saved PDB file so the sequence is taken directly from the structure.

from shared.target_utils import get_chain_sequence

fasta_lines = []

# Try to include the receptor chain sequence (required for AF2 multimer input)
pdb_file = WORKSPACE_DIR / f"{target_spec['pdb_id']}.pdb"
receptor_seq = None
if pdb_file.exists():
    try:
        pdb_text = pdb_file.read_text()
        receptor_seq = get_chain_sequence(pdb_text, target_spec["chain_id"])
        fasta_lines.append(f">{target_spec['pdb_id']}_{target_spec['chain_id']}_receptor")
        fasta_lines.append(receptor_seq)
        print(f"Receptor chain {target_spec['chain_id']}: {len(receptor_seq)} residues")
    except Exception as e:
        print(f"Could not extract receptor chain sequence: {e}")
        print("FASTA will contain peptides only. For AF2 multimer you will need to add the receptor chain manually.")
else:
    print(f"PDB file not found at {pdb_file}.")
    print("FASTA will contain peptides only. For AF2 multimer add the receptor chain sequence manually.")

# Include positive reference as a benchmark
ref_seq = target_spec["reference_policy"]["positive_control"]["sequence"]
fasta_lines.append(">reference_positive")
fasta_lines.append(ref_seq)

for c in shortlist:
    fasta_lines.append(f">{c['label']}")
    fasta_lines.append(c['sequence'])

fasta_text = "\n".join(fasta_lines) + "\n"
fasta_path = OUTPUT_DIR / "shortlist_candidates.fasta"
fasta_path.write_text(fasta_text)

print("\nFASTA file for strong-validation input:")
print(fasta_text)
print(f"Saved to {fasta_path}")
print()
print("Appropriate strong-validation lanes for peptides of this length:")
print("  - AlphaFold2 multimer: paste sequences into the standard AF2 Colab notebook")
print("  - Boltz-1: boltz predict shortlist_candidates.fasta --use_msa_server")
print("  - HADDOCK (web): https://wenmr.science.uu.nl/haddock2.4/ (free academic access)")
print("  - PyRosetta FlexPepDock: see intermediate-linux-gpu cookbook")
print()
print("Note: small-molecule docking tools (AutoDock Vina, Smina) are not appropriate")
print("for peptides of 8-20 residues. They don't model backbone flexibility and their")
print("scoring functions were not parameterised for peptides.")

## Step 6 — Generate Platform Upload Stubs

After running structure prediction (AF2 multimer, Boltz-1, HADDOCK, etc.) on the
FASTA file above, your candidates are ready to upload to the PeptideModel platform.

This cell writes one `card.yaml` stub per shortlisted candidate into
`workspace/promotion/upload_stubs/<label>/`. The stubs use the canonical upload
schema and `status: designed` — the correct status before structure prediction.

**What you must fill in before uploading:**
- `targets` — replace `fill-in-target-slug` with the platform slug (e.g. `gdf-8`)
- `metrics` — add ipTM and pLDDT from your structure-prediction run
- `structure_file` — copy your `.pdb` or `.cif` file here and update the filename
- `title` — write a human-readable descriptive name
- Update `status` from `designed` → `computed` once you have structure-prediction results

**What is already filled in:** sequence, source provenance (including heuristic score
and target PDB), scaffold/parent_card comment blocks, readme stub.

Do not invent new top-level fields. The platform schema accepts only the fields shown.

In [ ]:
import datetime as _dt

_now = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d")
_stubs_dir = OUTPUT_DIR / "upload_stubs"
_stubs_dir.mkdir(parents=True, exist_ok=True)

for c in shortlist:
    label = c['label']
    seq = c['sequence']
    score = c.get('composite_score') or 0.0
    stub_dir = _stubs_dir / label
    stub_dir.mkdir(exist_ok=True)

    card_yaml = f"""\
# card.yaml — upload stub for {label}
# Generated by colab-basics / notebook 05
# Status: INCOMPLETE — fill in sections marked FILL-IN before uploading.

title: "{label} — {target_spec['pdb_id']} target"  # FILL-IN: descriptive name
sequence: {seq}
targets:
  - fill-in-target-slug  # FILL-IN: platform slug, e.g. gdf-8, glp-1r, ghsr
# scaffold: fill-in-if-applicable   # uncomment if derived from a known scaffold
# parent_card: pep-XXXXX            # uncomment if derived from an existing platform card
status: designed  # change to 'computed' after running structure prediction

metrics: []
# FILL-IN after structure prediction — example:
# metrics:
#   - key: ipTM
#     value: 0.XXX
#     tool: Boltz-1
#   - key: pLDDT
#     value: 0.XXX
#     tool: Boltz-1

source:
  kind: other
  notes: >
    Designed by heuristic triage (colab-basics cookbook,
    heuristic_composition lane, score {score:.4f} — not a binding affinity).
    Target PDB: {target_spec['pdb_id']} chain {target_spec['chain_id']}.
    Promoted: {_now}.

structure_file: structure.pdb  # FILL-IN: copy your .pdb or .cif here after prediction
readme_file: readme.md
"""
    (stub_dir / "card.yaml").write_text(card_yaml)

    readme_md = f"""\
# {label}

**Sequence:** `{seq}`

**Target:** {target_spec['pdb_id']} chain {target_spec['chain_id']}

**Heuristic lane score:** {score:.4f} (composition heuristic — not a binding affinity)

**Pocket:** {target_spec.get('pocket_description', 'see target_spec.json')}

**Hypothesis:** {target_spec.get('mechanism_hypothesis', 'see target_spec.json')}

## Provenance

Promoted by `colab-basics` notebook 05 on {_now}.
Heuristic panel status was checked in notebook 02 before promotion.

## Next Step

1. Run structure prediction using `shortlist_candidates.fasta`
2. Fill in `metrics` in `card.yaml` with ipTM and pLDDT from your run
3. Copy the predicted structure file here and set `structure_file` in `card.yaml`
4. Update `status` from `designed` to `computed`
5. Upload via the PeptideModel platform upload form
"""
    (stub_dir / "readme.md").write_text(readme_md)

    print(f"  {label}/card.yaml + readme.md")

print(f"\nUpload stubs written to {_stubs_dir}")
print("These are pre-filled with provenance. Fill in targets, metrics, and structure_file")
print("after running structure prediction, then upload via the PeptideModel platform.")

## What Comes After This Cookbook

This cookbook covers **Claims Level 1** — cheap heuristic triage.

To make stronger claims, these are the next stages (not in this cookbook):

| Stage | Lane | What it adds |
|-------|------|-------------|
| Level 2 | Structure prediction (AF2, Boltz) | Interface geometry, pLDDT, ipTM |
| Level 3 | Target-decoy validation | Specificity: does the peptide prefer the real target over scrambled decoys? |
| Level 4 | Physics-based scoring (PyRosetta) | Interface energy, hotspot contacts |
| Level 5 | Corrected high-fidelity ranking | Combines all of the above with validated controls |

See `intermediate-linux-gpu/CLAIMS_LADDER.md` for the full ladder.

The `intermediate-linux-gpu/` cookbook in this repo covers these stages in
detail for Linux/GPU environments.

## Outputs

| File | Description |
|------|-------------|
| `workspace/promotion/shortlist.json` | Machine-readable promotion bundle with provenance |
| `workspace/promotion/promotion_summary.md` | Human-readable rationale and handoff protocol |
| `workspace/promotion/shortlist_candidates.fasta` | FASTA input for structure-prediction tools |
| `workspace/promotion/upload_stubs/<label>/card.yaml` | Platform upload stub per candidate (fill in after structure prediction) |
| `workspace/promotion/upload_stubs/<label>/readme.md` | Readme stub per candidate |

## Upload Flow

```
notebook 05
    └─ shortlist_candidates.fasta
           │
           ▼
    AF2 multimer / Boltz-1 / HADDOCK
           │
           ▼
    upload_stubs/<label>/
        ├─ card.yaml  ← fill in: targets, metrics (ipTM/pLDDT), structure_file
        │              ← update status: designed → computed
        └─ structure.pdb  ← copy predicted structure here
           │
           ▼
    PeptideModel platform upload
```

---

**You have completed the Colab Basics cookbook workflow.**

The upload stubs give you correctly-shaped `card.yaml` files that match the platform
contract. Run structure prediction on the FASTA, fill in the metrics and structure
file, then upload.